In [1]:
import bisect
import glob
import io
import os
import zipfile
from itertools import accumulate

import cv2
import numpy as np

# ---- where the competition archives live ---------------------------
BASE = "/kaggle/input/competitions/diabetic-retinopathy-detection"
print(sorted(os.listdir(BASE))[:8])

TARGET = 512


# ---- preprocessing: identical to every other notebook --------------
def stem(s):
    """'train/10003_left.jpeg' -> '10003_left'"""
    return os.path.splitext(os.path.basename(str(s)))[0]


def crop_fov(img, tol=7):
    """Trim the black surround around the circular field of view."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > tol
    if not mask.any():
        return img
    ys, xs = np.where(mask)
    return img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]


def pad_square(img):
    """Pad the short side so resizing does not distort the retina."""
    h, w = img.shape[:2]
    s = max(h, w)
    t, l = (s - h) // 2, (s - w) // 2
    return cv2.copyMakeBorder(img, t, s - h - t, l, s - w - l,
                              cv2.BORDER_CONSTANT, value=[0, 0, 0])


def preprocess(img, size=TARGET):
    """Full pipeline: crop, pad, resize. BGR in, BGR out."""
    return cv2.resize(pad_square(crop_fov(img)), (size, size),
                      interpolation=cv2.INTER_AREA)


# ---- split-archive reader ------------------------------------------
class SplitZipStream(io.RawIOBase):
    """Present train.zip.001..00N as a single seekable byte stream.

    zipfile reads the central directory from the END of the archive, then
    jumps to each member's offset. Both work as long as seeking is honoured,
    and no data is ever copied to disk.
    """

    def __init__(self, parts):
        self.parts = sorted(parts)
        self.fhs = [open(p, "rb") for p in self.parts]
        self.sizes = [os.path.getsize(p) for p in self.parts]
        self.starts = [0] + list(accumulate(self.sizes))
        self.total = self.starts[-1]
        self.pos = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET:
            self.pos = offset
        elif whence == io.SEEK_CUR:
            self.pos += offset
        elif whence == io.SEEK_END:
            self.pos = self.total + offset
        self.pos = max(0, min(self.pos, self.total))
        return self.pos

    def readinto(self, buf):
        want, got = len(buf), 0
        while got < want and self.pos < self.total:
            i = bisect.bisect_right(self.starts, self.pos) - 1
            local = self.pos - self.starts[i]
            self.fhs[i].seek(local)
            chunk = self.fhs[i].read(min(want - got, self.sizes[i] - local))
            if not chunk:
                break
            buf[got:got + len(chunk)] = chunk
            got += len(chunk)
            self.pos += len(chunk)
        return got

    def close(self):
        for f in self.fhs:
            f.close()
        super().close()


def open_split(pattern, buffer_size=1 << 22):
    """Open a split archive. BufferedReader on top is ~10x faster than raw."""
    parts = sorted(glob.glob(pattern))
    if not parts:
        raise FileNotFoundError(f"no parts matched {pattern}")
    return zipfile.ZipFile(io.BufferedReader(SplitZipStream(parts),
                                             buffer_size=buffer_size))


print("helpers ready:", [f.__name__ for f in
                         (stem, crop_fov, pad_square, preprocess, open_split)])

['sample.zip', 'sampleSubmission.csv.zip', 'test.zip.001', 'test.zip.002', 'test.zip.003', 'test.zip.004', 'test.zip.005', 'test.zip.006']
helpers ready: ['stem', 'crop_fov', 'pad_square', 'preprocess', 'open_split']


In [2]:
!rm -rf /kaggle/working/EyeQ
!git clone -q https://github.com/HzFu/EyeQ.git /kaggle/working/EyeQ
!git -C /kaggle/working/EyeQ log -1 --format="%H %ci"    # for provenance

1dcfabb08cd31045de1a4a3ee82e48f6833697aa 2026-06-29 11:14:49 +0800


In [3]:
import pandas as pd

EYEQ_REPO = "/kaggle/working/EyeQ"

tr_lab = pd.read_csv(f"{EYEQ_REPO}/data/Label_EyeQ_train.csv")
te_lab = pd.read_csv(f"{EYEQ_REPO}/data/Label_EyeQ_test.csv")

tr_lab["eyeq_split"] = "train"
te_lab["eyeq_split"] = "test"

eyeq = pd.concat([tr_lab, te_lab], ignore_index=True)
eyeq["stem"] = eyeq["image"].map(stem)

print(len(tr_lab), len(te_lab), len(eyeq))      # expect 12543, 16249, 28792
print(eyeq.groupby("eyeq_split").quality.value_counts().unstack())

12543 16249 28792
quality        0     1     2
eyeq_split                  
test        8471  4558  3220
train       8347  1876  2320


In [4]:
zf = open_split(f"{BASE}/train.zip.0*")
zf_test = open_split(f"{BASE}/test.zip.0*")     # 53 GB, takes a few minutes

lookup = {}
for mem in zf.namelist():
    if mem.lower().endswith(".jpeg"):
        lookup[stem(mem)] = ("train", mem)
for mem in zf_test.namelist():
    if mem.lower().endswith(".jpeg"):
        lookup[stem(mem)] = ("test", mem)

print(f"{len(lookup):,} images indexed across both archives")   # ~88,702

eyeq["src"] = eyeq["stem"].map(lookup)
print("unmatched:", eyeq["src"].isna().sum())                   # expect 0

88,702 images indexed across both archives
unmatched: 0


In [5]:
import os
import shutil
from pathlib import Path

OUT = "/kaggle/working/eyeq_512"
os.makedirs(OUT, exist_ok=True)

EXISTING = Path("/kaggle/input/datasets/izuchukwuoruche/eyeq-512-preprocessed/eyeq_512")

copied = 0
for p in EXISTING.iterdir():
    dest = f"{OUT}/{p.name}"
    if not os.path.exists(dest):
        shutil.copy2(p, dest)
        copied += 1

print(f"{copied:,} existing images copied")      # ~12,543
print(f"{len(os.listdir(OUT)):,} now present")

12,543 existing images copied
12,543 now present


In [6]:
import cv2
import numpy as np
from tqdm.auto import tqdm

ZF = {"train": zf, "test": zf_test}
todo = eyeq.dropna(subset=["src"])

# skip anything already on disk from the copy step
already = {os.path.splitext(f)[0] for f in os.listdir(OUT)}
todo = todo[~todo.stem.isin(already)]
print(f"{len(todo):,} images to convert")        # ~16,249

failed = []
for which in ("train", "test"):
    sub = todo[todo.src.map(lambda s: s[0]) == which]
    if sub.empty:
        continue

    # sequential read order within each archive - far faster than jumping
    # around across seven part files
    infos = sorted((ZF[which].getinfo(s[1]) for s in sub.src),
                   key=lambda i: i.header_offset)

    for info in tqdm(infos, desc=which):
        st = stem(info.filename)
        dest = f"{OUT}/{st}.png"
        if os.path.exists(dest):
            continue
        try:
            with ZF[which].open(info) as f:
                buf = np.frombuffer(f.read(), np.uint8)
            img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
            if img is None:
                raise ValueError("decode failed")
            cv2.imwrite(dest, preprocess(img))
        except Exception as e:
            failed.append((st, str(e)))

print(f"\n{len(os.listdir(OUT)):,} images total")    # expect 28,792
print(f"failed: {len(failed)}")
if failed:
    print(failed[:10])

16,249 images to convert


test:   0%|          | 0/16249 [00:00<?, ?it/s]


28,792 images total
failed: 0


In [7]:
!du -sh {OUT}
!df -h /kaggle/working

6.8G	/kaggle/working/eyeq_512
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  6.8G   13G  35% /kaggle/working


In [8]:
rows = []
for _, r in eyeq.dropna(subset=["src"]).iterrows():
    if not os.path.exists(f"{OUT}/{r.stem}.png"):
        continue
    patient, eye = r.stem.rsplit("_", 1)
    rows.append({
        "stem": r.stem,
        "image": r.image,
        "quality": int(r.quality),
        "patient_id": patient,
        "eye": eye,
        "eyeq_split": r.eyeq_split,
        "path": f"eyeq_512/{r.stem}.png",
    })

manifest = pd.DataFrame(rows)
manifest.to_csv("/kaggle/working/manifest_eyeq_512.csv", index=False)

print(manifest.shape)
print(manifest.eyeq_split.value_counts().to_dict())
print(manifest.groupby("eyeq_split").quality.value_counts().unstack())

(28792, 7)
{'test': 16249, 'train': 12543}
quality        0     1     2
eyeq_split                  
test        8471  4558  3220
train       8347  1876  2320


In [9]:
spread = manifest.groupby("patient_id")["eyeq_split"].nunique()
overlap = spread[spread > 1]

print(f"patients spanning EyeQ's train/test boundary: {len(overlap)}")
print(f"total patients: {manifest.patient_id.nunique():,}")

if len(overlap):
    n_imgs = manifest[manifest.patient_id.isin(overlap.index)].shape[0]
    print(f"affected images: {n_imgs:,}")
    print("examples:", list(overlap.index[:5]))
else:
    print("clean - EyeQ's partition is patient-disjoint")

patients spanning EyeQ's train/test boundary: 0
total patients: 20,662
clean - EyeQ's partition is patient-disjoint


In [10]:
shutil.rmtree("/kaggle/working/EyeQ", ignore_errors=True)
shutil.rmtree("/kaggle/working/.virtual_documents", ignore_errors=True)

print(sorted(os.listdir("/kaggle/working")))
!du -sh /kaggle/working

['__notebook__.ipynb', 'eyeq_512', 'manifest_eyeq_512.csv']
6.8G	/kaggle/working
